# VisionSetil Large Public Fungi Benchmark

This notebook orchestrates the batch evaluation of the **VisionSetil** mushroom identification pipeline against large, public datasets (such as FungiCLEF 2025, FungiTastic, or Danish Fungi 2020) on Kaggle. It converts the schemas, applies sampling, processes the evaluation pipeline, and outputs reports.

## 1. Environment Check
Inspect the Kaggle runtime resources and verify GPU availability.

In [ ]:
!nvidia-smi
import torch
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))

## 2. Attach Large Dataset & Install Dependencies
Download the VisionSetil source code, install system requirements, and attach the target dataset to `/kaggle/input`.

In [ ]:
# Clone the codebase
!git clone https://github.com/AlonsoAlviraa/VisionSetil.git
%cd VisionSetil

# Install requirements
!pip install -r backend/requirements.txt
!pip install psutil gputil

## 3. Inspect Dataset
Execute the dataset inspection tool to analyze column availability, metadata structure, and row formats.

In [ ]:
!python kaggle/inspect_kaggle_dataset.py \
  --dataset-root /kaggle/input/fungi-clef-2025

## 4. Convert Dataset to VisionSetil Format
The FungiCLEF/FungiTastic/DF20 datasets are mounted under `/kaggle/input/`. We'll run a converter to translate taxonomy, image paths, and environmental metadata into our standard validation format.

In [ ]:
# Execute a conversion check with a config example
import json
config_file = "kaggle/configs/fungiclef2025_config.example.json"
with open(config_file, "r") as f:
    print(json.dumps(json.load(f), indent=2))

## 5. Run Benchmark on Subset (Smoke Test)
Execute a quick validation test on a small subset (e.g. 50 cases) to verify that paths resolve and scripts execute correctly.

In [ ]:
!python kaggle/run_large_dataset_benchmark.py \
  --config kaggle/configs/fungiclef2025_1000_real_models_config.json \
  --max-cases 50 \
  --shuffle \
  --seed 42 \
  --include-dangerous-genera \
  --debug-safety

## 6. Run Larger Benchmark
Execute the benchmark against a larger population with risk-balanced sampling to evaluate both accuracy and safety triggers.

In [ ]:
# Medium test - 200 cases
!python kaggle/run_large_dataset_benchmark.py \
  --config kaggle/configs/fungiclef2025_1000_real_models_config.json \
  --max-cases 200 \
  --shuffle \
  --seed 42 \
  --include-dangerous-genera

# Full test - 1000 cases
!python kaggle/run_large_dataset_benchmark.py \
  --config kaggle/configs/fungiclef2025_1000_real_models_config.json \
  --max-cases 1000 \
  --shuffle \
  --seed 42 \
  --include-dangerous-genera

## 7. Summarize Metrics
Display the general evaluation reports from the run.

In [ ]:
!python eval/scripts/summarize_eval.py --report /kaggle/working/visionsetil_outputs/real_report.json

## 8. Dangerous Cases
Verify the safety policy alignment on dangerous and poisonous genera (e.g. Amanita, Galerina).

In [ ]:
import pandas as pd
import json
from pathlib import Path

fails_path = Path("/kaggle/working/visionsetil_outputs/dangerous_failures.json")
if fails_path.exists():
    with open(fails_path, "r") as f:
        data = json.load(f)
    print(f"Found {len(data)} dangerous classification failures:")
    if data:
        display(pd.DataFrame(data).head(10))
else:
    print("No dangerous failures found.")

## 9. Mock vs Real Warning
Display the large dataset summary generated, highlighting whether calculations are emulated via mocks or executed with real weights.

In [ ]:
summary_path = Path("/kaggle/working/visionsetil_outputs/large_dataset_summary.md")
if summary_path.exists():
    with open(summary_path, "r", encoding="utf-8") as f:
        print(f.read())
else:
    print("Summary report file not found.")

## 10. Download Outputs
Review outputs ready for download.

In [ ]:
output_dir = Path("/kaggle/working/visionsetil_outputs")
if output_dir.exists():
    print("Artifacts in output folder:")
    for file in output_dir.glob("*"):
        print(f"  - {file.name} ({file.stat().st_size} bytes)")
else:
    print("Output folder does not exist.")